# 05. Pre-Publish Prediction Logging

Log every video you plan to upload **before you publish it**. The model outputs a predicted view count WITH a confidence interval. After publishing and waiting 7–14 days, use Notebook 06 to measure prediction accuracy.

In [1]:
import pandas as pd
import numpy as np
import os, sys, joblib

sys.path.append('..')
from src.recommender import YouTubeRecommender

In [2]:
# Load model bundle
recommender = YouTubeRecommender.load('../src/model.joblib')
print("✅ Recommender loaded!")

✅ Recommender loaded!


In [3]:
LOG_PATH = '../predictions/predictions_log.csv'
os.makedirs('../predictions', exist_ok=True)

def log_prediction(channel_name, title, duration_minutes, publish_hour_ist,
                   publish_day_of_week=2, niche='personal_finance', subscribers=0,
                   description="", tags="[]", voice_type="human_own",
                   presentation_style="face_presenter", is_highly_edited=1):
    """
    Log a pre-publish video prediction.
    """
    result = recommender.predict(
        title=title,
        duration_minutes=duration_minutes,
        publish_hour_ist=publish_hour_ist,
        publish_day_of_week=publish_day_of_week,
        niche=niche,
        subscribers=subscribers,
        voice_type=voice_type,
        presentation_style=presentation_style,
        is_highly_edited=is_highly_edited,
        description=description,
        tags=tags,
    )
    recommender.print_report(result)
    
    # Use 30-day forecast as standard
    pred_30 = result['predictions'][30]
    
    entry = {
        'prediction_date':              pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
        'channel_name':                 channel_name,
        'title':                        title,
        'voice_type':                   voice_type,
        'presentation_style':           presentation_style,
        'is_highly_edited':             is_highly_edited,
        'duration_minutes':             duration_minutes,
        'publish_hour_ist':             publish_hour_ist,
        'publish_day_of_week':          publish_day_of_week,
        'niche':                        niche,
        'subscribers_at_prediction':    subscribers,
        'predicted_virality_multiplier': pred_30['v_ratio'],
        'predicted_views':              pred_30['predicted_views'],
        'views_low_80pct':              pred_30['views_low'],
        'views_high_80pct':             pred_30['views_high'],
        'actual_views':                 np.nan,
        'actual_multiplier':            np.nan,
        'within_confidence_interval':   np.nan,
        'published_video_id':           '',
        'is_revalidated':               0,
    }
    
    if os.path.exists(LOG_PATH):
        log_df = pd.read_csv(LOG_PATH)
        log_df = pd.concat([log_df, pd.DataFrame([entry])], ignore_index=True)
    else:
        log_df = pd.DataFrame([entry])
    
    log_df.to_csv(LOG_PATH, index=False)
    print(f"\n✅ Logged: '{title[:50]}...'")
    return entry

In [4]:
# ============================================================
# Log your upcoming videos here — before publishing!
# ============================================================

# Experimental Channel A: India Data Stories (Geopolitics & Economy)
log_prediction(
    channel_name       = 'India Data Stories',
    title              = 'Why India is Running Out of Water: 5 Shocking Facts',
    voice_type         = 'human_own',
    presentation_style  = 'faceless_broll',
    is_highly_edited   = 1,
    duration_minutes   = 12.5,
    publish_hour_ist   = 19,
    publish_day_of_week = 2,  # Wednesday
    niche              = 'geopolitics_economy',
    subscribers        = 15000,
    description        = 'India water crisis explained. #IndiaWater #Economy #Facts',
)

print("\n" + "="*56 + "\n")

# Experimental Channel B: Student Money Academy (Personal Finance)
log_prediction(
    channel_name       = 'Student Money Academy',
    title              = '5 Finance Secrets Schools Never Teach You in India',
    voice_type         = 'human_own',
    presentation_style  = 'face_presenter',
    is_highly_edited   = 1,
    duration_minutes   = 11.0,
    publish_hour_ist   = 18,
    publish_day_of_week = 3,  # Thursday
    niche              = 'personal_finance',
    subscribers        = 8000,
    description        = 'Student personal finance tips. #StudentFinance #Money #India',
)


                   CREATOR FORECASTING PLAYBOOK
 Proposed Title : Why India is Running Out of Water: 5 Shocking Facts
 Niche          : Geopolitics & Economy
 Format/Style   : FACELESS_BROLL | HUMAN_OWN | HIGHLY EDITED
 Sub Baseline   : 1,500.0 views (preceding uploads median)
-----------------------------------------------------------------
 [1] VIEWS & VELOCITY HORIZONS
  Virality Tier : Baseline (Normal channel performance)
  Horizon    | Expected Views   | 80% Confidence Band    | V-Ratio 
  ---------- | ---------------- | ---------------------- | --------
  7  Days     | 1,282            | 152 - 10,737 | 0.85x
  30 Days     | 1,254            | 148 - 10,502 | 0.84x
  90 Days     | 1,196            | 142 - 10,022 | 0.80x
-----------------------------------------------------------------
 [2] TITLE ANALYSIS & REFRAMING
  Character Count  : 51 (Ideal: 45-65)
  Sentiment Tone   : Polarity -1.00 | Subjectivity 1.00

  Advisory Warnings:
    ✔ Add a curiosity question (e.g. 'Is this...?

{'prediction_date': '2026-05-21 20:07:59',
 'channel_name': 'Student Money Academy',
 'title': '5 Finance Secrets Schools Never Teach You in India',
 'voice_type': 'human_own',
 'presentation_style': 'face_presenter',
 'is_highly_edited': 1,
 'duration_minutes': 11.0,
 'publish_hour_ist': 18,
 'publish_day_of_week': 3,
 'niche': 'personal_finance',
 'subscribers_at_prediction': 8000,
 'predicted_virality_multiplier': 0.5,
 'predicted_views': 400,
 'views_low_80pct': 46,
 'views_high_80pct': 3355,
 'actual_views': nan,
 'actual_multiplier': nan,
 'within_confidence_interval': nan,
 'published_video_id': '',
 'is_revalidated': 0}

In [5]:
# View current log
log_df = pd.read_csv(LOG_PATH)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
print(f"\nTotal logged predictions: {len(log_df)}")
print(log_df[['channel_name', 'title', 'voice_type', 'predicted_views', 
              'views_low_80pct', 'views_high_80pct', 'is_revalidated']].to_string())


Total logged predictions: 2
            channel_name                                                title voice_type  predicted_views  views_low_80pct  views_high_80pct  is_revalidated
0     India Data Stories  Why India is Running Out of Water: 5 Shocking Facts  human_own             1254              148             10502               0
1  Student Money Academy   5 Finance Secrets Schools Never Teach You in India  human_own              400               46              3355               0
